# TP NERO bimanuel — MuJoCo / Google Colab

Ce notebook utilise **votre modèle NERO**, issu du dossier `models/models/agx_arm_urdf/nero_dual_arm`.
Le modèle source choisi est **nero_dual_arm.mpc.xml** : deux bras de 7 articulations et deux
glissières par pince, soit **18 actionneurs**. Le robot reste fixé au sol.

1. Importer ce notebook dans [Google Colab](https://colab.research.google.com/).
2. Utiliser une session **CPU neuve**, puis **Exécution → Tout exécuter**.
3. À la cellule d’import, sélectionner **NERO_Colab_assets.zip**, fourni avec ce notebook.

L’archive contient les maillages NERO de votre dossier et le pot French’s YCB. Le notebook
installe MuJoCo, génère une table et le pot, affiche une vidéo, résout l’IK de pose de chaque bras,
puis ferme une pince par impédance, soulève le pot et le maintient trois secondes en hauteur.
L’autre bras est commandé en position d’attente.
`ARM = 'right'` ou `'left'` choisit le bras qui saisit le pot ; **les deux bras sont présents et commandés**.
Ce TP ne réalise pas un passage du pot entre les bras ni une saisie simultanée à deux pinces.

Tests effectués localement sous Windows avec MuJoCo 3.3.7. L’exécution sur un compte Colab
n’a pas été effectuée ici. La session Colab utilise OSMesa pour le rendu CPU sans fenêtre.


In [ ]:
import subprocess, sys, os
subprocess.run(['apt-get', '-qq', 'update'], check=True)
subprocess.run(['apt-get', '-qq', 'install', '-y', 'libosmesa6'], check=True)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'mujoco==3.3.7', 'numpy>=1.26,<3', 'scipy>=1.11,<2',
    'imageio>=2.34,<3', 'imageio-ffmpeg==0.6.0', 'matplotlib>=3.7'])
os.environ['MUJOCO_GL'] = 'osmesa'
os.environ['PYOPENGL_PLATFORM'] = 'osmesa'
print('MuJoCo installé. Rendu CPU OSMesa configuré.')

## 1. Importer les fichiers de votre robot

Le modèle ne dépend d’aucun chemin Windows dans Colab. Les fichiers sont extraits localement
dans la session. Le code vérifie l’intégrité de l’archive attendue avant extraction.
Les licences du robot et du pot sont incluses. Votre dossier original n’a pas été modifié.


In [ ]:
from pathlib import Path
import zipfile, hashlib
archive = Path('NERO_Colab_assets.zip')
if not archive.exists():
    from google.colab import files
    uploaded = files.upload()  # Choisir NERO_Colab_assets.zip
    if archive.name not in uploaded:
        raise FileNotFoundError('Sélectionner le fichier NERO_Colab_assets.zip fourni.')
expected_sha256 = 'b27d6703c758329e2ba54929c4dbdbab5fe170a7b1a11c7ccaa92758bdd97c36'
if hashlib.sha256(archive.read_bytes()).hexdigest() != expected_sha256:
    raise ValueError('Archive différente : utiliser celle fournie avec ce notebook.')
destination = Path.cwd().resolve()
with zipfile.ZipFile(archive) as z:
    for info in z.infolist():
        target = (destination / info.filename).resolve()
        if not target.is_relative_to(destination):
            raise ValueError('Chemin non autorisé dans l’archive.')
    z.extractall(destination)
print('Modèles NERO et French’s disponibles.')

## 2. Repères des pinces et conventions

Le fichier MuJoCo fourni n’a pas de site TCP. Nous ajoutons :

| Bras | Site de commande | Corps parent |
|---|---|---|
| Droit | `right_grasp_center` | `right_link7` |
| Gauche | `left_grasp_center` | `left_link7` |

Dans le repère `link7`, chaque site a pour position approximative **[0,175 ; 0 ; −0,0235] m**.
C’est l’origine commune des deux glissières à fermeture nulle. Les axes sont ceux de `gripper_base`,
avec le quaternion MuJoCo **wxyz ≈ [0,5 ; −0,5 ; 0,5 ; −0,5]**.
Ces valeurs sont tirées des transformations du modèle fourni, sans reprendre le décalage du Panda.
Le point est un **TCP nominal**, pas un centre de contact mesuré ; sa calibration physique reste à faire.

+z du TCP pointe vers les doigts, y est l’axe d’ouverture. Les axes visibles sont x rouge,
y vert, z bleu. `lab.transform(side)` donne **T_world_tcp**.
Les orientations cibles sont des matrices 3 × 3 exprimées dans le monde, les positions en mètres.

**Attention au signe des doigts NERO :** fermé `(0, 0)` ; ouvert `(+0,05, −0,05)` m.
L’ouverture nominale totale est donc **q₁ − q₂**, entre 0 et 0,10 m.
Pour une ouverture désirée w, les consignes sont `(w/2, −w/2)`.

Chaque moteur de pince applique :

$$F_i = operatorname{clip}left(K(q_{i,d}-q_i)-Ddot q_i,;-F_{max},;F_{max}
ight).$$

K = 300 N/m, D = 5 N·s/m, Fmax = 8 N par doigt (limite du modèle : 10 N).
Le bras reçoit un PD articulaire et la compensation des efforts de biais, limités à ±16 N·m
comme dans le fichier source. Ces valeurs décrivent le modèle de recherche fourni.


## 3. Adaptation du modèle et contrôleur Python

Les collisions du modèle source sont désactivées. Pour ce TP, les copies de géométrie
destinées aux collisions sont activées. MuJoCo utilise leurs **enveloppes convexes** : elles
restent une approximation de la géométrie réelle, surtout pour les formes concaves.
Deux paires de carters voisins par bras sont exclues (`root`/`link1`, `link5`/`link7`), car leurs
maillages se chevauchent dans l’assemblage fourni. Les contacts robot–table, robot–pot
et entre les bras restent actifs. Les contacts non attendus dépassant 1 mm sont signalés.

Le pas est 1 ms, le solveur Newton, l’intégrateur `implicitfast`. Pour réduire la dérive
tangentielle des contacts souples : `cone='elliptic'`, `impratio=10`, `noslip_iterations=20`.
Ces réglages numériques ne sont pas une calibration physique des matériaux ; voir la
[documentation MuJoCo](https://mujoco.readthedocs.io/en/3.2.5/modeling.html).
Les capteurs bruités inutilisés
du fichier source sont retirés. Les limites, inerties et échelles des maillages sont conservées.

L’IK utilise une optimisation bornée avec plusieurs initialisations ; elle ne modifie pas
l’état du robot simulé. Les consignes sont ensuite interpolées et suivies par la dynamique.
Elle ne planifie pas l’évitement des obstacles. Les tests de contacts ne remplacent pas
un planificateur anticollision, surtout si vous modifiez les cibles.


In [ ]:
%%writefile lift_metrics.py
"""Measure a real free-body lift independently of the gripper target."""
import numpy as np
import mujoco

class LiftMonitor:
    def __init__(self, model, data, object_name='mustard_bottle', table_name='table'):
        self.model, self.data = model, data
        self.body = model.body(object_name).id
        self.table = model.geom(table_name).id
        # Use the rendered bottle mesh, including the cap, for bottom clearance.
        self.visual = model.geom('mustard_bottle_visual_geom').id
        mesh = model.geom_dataid[self.visual]
        start, count = model.mesh_vertadr[mesh], model.mesh_vertnum[mesh]
        self.vertices = model.mesh_vert[start:start+count].copy()
        self.table_top = float(model.geom_pos[self.table,2] + model.geom_size[self.table,2])

    def sample(self, tcp_position, tcp_rotation):
        m,d = self.model,self.data
        rotation = d.geom_xmat[self.visual].reshape(3,3)
        bottom = float(np.min(self.vertices @ rotation[2,:]) + d.geom_xpos[self.visual,2])
        table_contact = False
        for c in d.contact:
            if self.table in (c.geom1,c.geom2):
                other = c.geom2 if c.geom1 == self.table else c.geom1
                if m.geom_bodyid[other] == self.body:
                    table_contact = True
        return dict(object_z_m=float(d.xpos[self.body,2]),
                    clearance_m=bottom-self.table_top,
                    object_in_tcp=(tcp_rotation.T @ (d.xpos[self.body]-tcp_position)).tolist(),
                    table_contact=table_contact)

def assess_lift(before, hold, final, *, required_rise=.10):
    """Require rise, table separation, persistent contact and a stable 3 s hold.

    hold contains timestamped samples, each with telemetry and contact forces.
    Heights are measured on the OBJECT, never inferred from a joint/TCP target.
    """
    if len(hold) < 2:
        raise ValueError('Need hold samples to evaluate lift stability.')
    offsets = np.array([row['object_in_tcp'] for row in hold])
    drift = float(np.max(np.linalg.norm(offsets-offsets[0],axis=1)))
    clearance = min(row['clearance_m'] for row in hold)
    rise = final['object_z_m']-before['object_z_m']
    separation = not any(row['table_contact'] for row in hold) and not final['table_contact']
    contact_fraction = float(np.mean([np.all(np.array(row['contact'])>.1) for row in hold]))
    duration = hold[-1]['time']-hold[0]['time']
    slip = float(np.linalg.norm(np.array(final['object_in_tcp'])-before['object_in_tcp']))
    success = (rise >= required_rise and clearance >= .08 and separation and
               contact_fraction >= .99 and drift <= .01 and duration >= 2.9)
    return dict(lift_success=bool(success), object_lift_m=float(rise),
                final_clearance_m=float(final['clearance_m']),
                min_hold_clearance_m=float(clearance), hold_relative_drift_m=drift,
                total_relative_slip_m=slip, hold_bilateral_contact_fraction=contact_fraction,
                hold_duration_measured_s=float(duration), table_separated=bool(separation))


In [ ]:
%%writefile nero_lab.py
"""NERO dual arm from the user's models: pose IK and finger impedance."""
from pathlib import Path
import copy
import json
import xml.etree.ElementTree as ET
import numpy as np
import mujoco
from scipy.spatial.transform import Rotation
from scipy.optimize import least_squares
from lift_metrics import LiftMonitor, assess_lift

SIDES = ('right', 'left')
TABLE_Z = .40
DOWN = np.array([[0., 1., 0.], [1., 0., 0.], [0., 0., -1.]])
TCP_POS = np.array([.175, -5.28942e-7, -.0235005])
TCP_QUAT = np.array([.499998, -.5, .500002, -.5])  # wxyz, same axes as gripper_base

def build_scene(root='nero_assets', mustard='assets/mustard', output='nero_scene.xml', side='right'):
    if side not in SIDES:
        raise ValueError(side)
    root, mustard = Path(root).resolve(), Path(mustard).resolve()
    tree = ET.parse(root / 'nero_dual_arm.mpc.xml')
    xml = tree.getroot()
    xml.find('compiler').attrib.pop('meshdir', None)
    xml.find('option').attrib.update(timestep='.001', integrator='implicitfast', solver='Newton', iterations='80',
                                    cone='elliptic', impratio='10', noslip_iterations='20')
    # Sensor noise attributes in older generated MJCF are unnecessary for this TP.
    if xml.find('sensor') is not None:
        xml.remove(xml.find('sensor'))
    for mesh in xml.findall('./asset/mesh'):
        mesh.set('file', (root / 'meshes' / mesh.get('file')).as_posix())
    world = xml.find('worldbody')
    contact = ET.SubElement(xml, 'contact')
    # Touching/overlapping housings in the supplied CAD, not obstacles.
    for side_name in SIDES:
        ET.SubElement(contact, 'exclude', body1='root', body2=f'{side_name}_link1')
        ET.SubElement(contact, 'exclude', body1=f'{side_name}_link5', body2=f'{side_name}_link7')
    for geom in world.iter('geom'):
        if geom.get('rgba') == '':
            geom.attrib.pop('rgba')
        if geom.get('class') == 'visualgeom':
            geom.set('rgba', '.55 .60 .67 1')
            geom.set('group', '1')
        else:
            # Activate the collision copies supplied with the model.
            # MuJoCo uses their convex hulls: this is an approximation.
            geom.attrib.update(contype='1', conaffinity='1', group='3',
                               friction='1 .01 .001', solref='.01 1', solimp='.95 .99 .001')
    world.find("geom[@name='ground']").attrib.update(size='3 3 .1', group='0', rgba='.18 .22 .28 1')
    for side_name in SIDES:
        wrist = xml.find(f".//body[@name='{side_name}_link7']")
        ET.SubElement(wrist, 'site', name=f'{side_name}_grasp_center',
                      pos=' '.join(map(str, TCP_POS)), quat=' '.join(map(str, TCP_QUAT)),
                      size='.004', rgba='0 1 1 1', group='4')
        rotation = Rotation.from_quat(TCP_QUAT[[1, 2, 3, 0]]).as_matrix()
        for i, color in enumerate(['1 0 0 1', '0 1 0 1', '0 .3 1 1']):
            end = TCP_POS + .06 * rotation[:, i]
            ET.SubElement(wrist, 'site', type='cylinder', size='.0015', group='4', rgba=color,
                          fromto=' '.join(map(str, np.r_[TCP_POS, end])))
    ET.SubElement(world, 'geom', name='table', type='box', pos='0 -.5 .375', size='.7 .35 .025',
                  rgba='.62 .47 .32 1', contype='1', conaffinity='1', friction='1 .01 .001', group='0')
    for x in [-.6, .6]:
        for y in [-.75, -.25]:
            ET.SubElement(world, 'geom', type='box', pos=f'{x} {y} .175', size='.025 .025 .175',
                          rgba='.3 .3 .3 1', contype='1', conaffinity='1', group='0')
    source = ET.parse(mustard / 'source.xml').getroot()
    for asset in source.find('asset'):
        asset = copy.deepcopy(asset)
        if asset.get('file'):
            asset.set('file', (mustard / asset.get('file')).as_posix())
        xml.find('asset').append(asset)
    obj = copy.deepcopy(source.find('worldbody/body'))
    x = -.25 if side == 'right' else .25
    obj.set('pos', f'{x} -.45 {TABLE_Z + .07830708 + .001}')
    for geom in obj.findall('geom'):
        # Override disabled default contacts inherited from the robot XML.
        is_visual = geom.get('contype') == '0'
        geom.attrib.update(contype='0' if is_visual else '1', conaffinity='0' if is_visual else '1',
                           friction='1.2 .01 .001', solref='.01 1', solimp='.95 .99 .001')
    world.append(obj)
    visual = ET.SubElement(xml, 'visual')
    ET.SubElement(visual, 'global', offwidth='960', offheight='720')
    ET.indent(tree)
    tree.write(output, encoding='utf-8', xml_declaration=True)
    return Path(output)

class NeroLab:
    def __init__(self, scene='nero_scene.xml'):
        self.model = m = mujoco.MjModel.from_xml_path(str(scene))
        self.data = mujoco.MjData(m)
        self.arms = {}
        for side in SIDES:
            names = [f'{side}_joint{i}' for i in range(1, 8)]
            fingers = [f'{side}_gripper_joint{i}' for i in [1, 2]]
            joint_ids = np.array([m.joint(n).id for n in names])
            fids = np.array([m.joint(n).id for n in fingers])
            self.arms[side] = dict(q=m.jnt_qposadr[joint_ids], v=m.jnt_dofadr[joint_ids],
                u=np.array([m.actuator(n).id for n in names]), limits=m.jnt_range[joint_ids],
                fq=m.jnt_qposadr[fids], fv=m.jnt_dofadr[fids],
                fu=np.array([m.actuator(n).id for n in fingers]), site=m.site(f'{side}_grasp_center').id,
                fingers=[m.body(f'{side}_gripper_link{i}').id for i in [1, 2]])
        self.object_body = m.body('mustard_bottle').id
        self.lift_monitor = LiftMonitor(m, self.data)
        self.kp = np.array([160., 160., 140., 140., 80., 60., 45.])
        self.kd = np.array([18., 18., 16., 16., 8., 6., 5.])
        self.k_finger, self.d_finger, self.force_limit = 300., 5., 8.
        self.reset()

    def reset(self):
        mujoco.mj_resetData(self.model, self.data)
        self.refs = {side: np.zeros(7) for side in SIDES}
        self.openings = {side: .10 for side in SIDES}
        for arm in self.arms.values():
            self.data.qpos[arm['fq']] = [.05, -.05]
        mujoco.mj_forward(self.model, self.data)
        self.log = []
        self.unexpected_contacts = set()

    def pose(self, side, data=None):
        d = self.data if data is None else data
        site = self.arms[side]['site']
        return d.site_xpos[site].copy(), d.site_xmat[site].reshape(3, 3).copy()

    def transform(self, side):
        p, r = self.pose(side)
        result = np.eye(4)
        result[:3, :3], result[:3, 3] = r, p
        return result

    def ik(self, side, position, rotation, seed=None):
        """Bounded 6D numerical IK; does not modify simulation state."""
        p, r = np.asarray(position, float), np.asarray(rotation, float)
        if p.shape != (3,) or r.shape != (3, 3) or not np.isfinite(p).all() or not np.isfinite(r).all():
            raise ValueError('Expected finite position (3,) and rotation (3,3).')
        if not np.allclose(r.T @ r, np.eye(3), atol=1e-6) or not np.isclose(np.linalg.det(r), 1):
            raise ValueError('Expected a proper rotation matrix.')
        arm = self.arms[side]
        work = mujoco.MjData(self.model)
        work.qpos[:] = self.data.qpos
        def residual(q):
            work.qpos[arm['q']] = q
            mujoco.mj_forward(self.model, work)
            actual_p, actual_r = self.pose(side, work)
            return np.r_[actual_p-p, .3*Rotation.from_matrix(r @ actual_r.T).as_rotvec()]
        low, high = arm['limits'][:, 0]+.002, arm['limits'][:, 1]-.002
        seed = self.refs[side] if seed is None else np.asarray(seed)
        seeds = [np.clip(seed, low, high)]
        rng = np.random.default_rng(23)
        seeds += [rng.uniform(low*.7, high*.7) for _ in range(10)]
        best = np.inf
        for start in seeds:
            answer = least_squares(residual, start, bounds=(low, high), max_nfev=180,
                                   ftol=1e-9, xtol=1e-9, gtol=1e-9)
            e = residual(answer.x)
            best = min(best, np.linalg.norm(e))
            if np.linalg.norm(e[:3]) < 2e-4 and np.linalg.norm(e[3:])/.3 < .002:
                return answer.x
        raise RuntimeError(f'IK {side} impossible for this pose (weighted residual {best:.4g}).')

    def control(self):
        if not 0 < self.force_limit <= 10:
            raise ValueError('Finger force_limit must be in (0,10] N, as in the source model.')
        m, d = self.model, self.data
        for side, a in self.arms.items():
            tau = self.kp*(self.refs[side]-d.qpos[a['q']])-self.kd*d.qvel[a['v']]+d.qfrc_bias[a['v']]
            d.ctrl[a['u']] = np.clip(tau, -16, 16)
            # Source model: positive slide 1, NEGATIVE slide 2 when open.
            target = np.array([.5, -.5]) * self.openings[side]
            force = self.k_finger*(target-d.qpos[a['fq']])-self.d_finger*d.qvel[a['fv']]
            d.ctrl[a['fu']] = np.clip(force, -self.force_limit, self.force_limit)

    def contact_forces(self, side):
        forces, wrench = np.zeros(2), np.zeros(6)
        for i in range(self.data.ncon):
            c = self.data.contact[i]
            bodies = [self.model.geom_bodyid[c.geom1], self.model.geom_bodyid[c.geom2]]
            if self.object_body not in bodies:
                continue
            mujoco.mj_contactForce(self.model, self.data, i, wrench)
            for j, body in enumerate(self.arms[side]['fingers']):
                if body in bodies:
                    forces[j] += max(0., wrench[0])
        return forces

    def segment(self, targets=None, openings=None, duration=2., capture=None, phase='move'):
        targets = {} if targets is None else targets
        openings = {} if openings is None else openings
        if any(not 0 <= w <= .10 for w in openings.values()):
            raise ValueError('Total opening must be in [0, 0.10] m.')
        starts = {s:q.copy() for s,q in self.refs.items()}
        initial_openings = self.openings.copy()
        steps = max(1, round(duration/self.model.opt.timestep))
        for step in range(steps):
            u = (step+1)/steps
            blend = 10*u**3-15*u**4+6*u**5
            for side, target in targets.items():
                self.refs[side] = starts[side] + blend*(target-starts[side])
            for side, target in openings.items():
                self.openings[side] = initial_openings[side]+blend*(target-initial_openings[side])
            self.control()
            mujoco.mj_step(self.model, self.data)
            # Record obstacle contacts beyond the intended pot/table/finger contacts.
            for contact in self.data.contact:
                if contact.dist >= -.001:
                    continue
                b1, b2 = [self.model.geom_bodyid[g] for g in [contact.geom1, contact.geom2]]
                names = [self.model.body(b).name for b in [b1, b2]]
                intended = (self.object_body in [b1, b2] and
                            any(n == 'world' or 'gripper_link' in n for n in names))
                if not intended:
                    self.unexpected_contacts.add(tuple(sorted(names)))
            if not np.isfinite(self.data.qpos).all():
                raise RuntimeError('Unstable simulation')
            if step % 40 == 0:
                mujoco.mj_forward(self.model, self.data)
                row = dict(time=float(self.data.time), phase=phase)
                for side, a in self.arms.items():
                    qf = self.data.qpos[a['fq']]
                    row[side] = dict(opening=float(qf[0]-qf[1]), contact=self.contact_forces(side).tolist(),
                                     force=self.data.ctrl[a['fu']].tolist(),
                                     **self.lift_monitor.sample(*self.pose(side)))
                self.log.append(row)
            if capture is not None:
                capture(self)
        mujoco.mj_forward(self.model, self.data)

    def reach_poses(self, poses, duration=3., capture=None):
        targets = {side:self.ik(side, *pose) for side,pose in poses.items()}
        self.segment(targets=targets, duration=duration, capture=capture)
        self.segment(duration=.6, capture=capture, phase='settle')
        errors = {}
        for side, (p, r) in poses.items():
            actual_p, actual_r = self.pose(side)
            e = dict(position_m=float(np.linalg.norm(actual_p-p)),
                     orientation_rad=float(Rotation.from_matrix(r @ actual_r.T).magnitude()))
            errors[side] = e
            if e['position_m'] > .01 or e['orientation_rad'] > .08:
                raise RuntimeError(f'Tracking failed for {side}: {e}')
        return errors

    def demo(self, side='right', capture=None):
        self.reset()
        self.segment(duration=.5, capture=capture, phase='settle')
        object_pos = self.data.xpos[self.object_body].copy()
        grasp = np.array([object_pos[0], object_pos[1], TABLE_Z+.16])
        standby = np.array([.45 if side=='right' else -.45, -.30, .8])
        other = 'left' if side=='right' else 'right'
        self.reach_poses({side:(grasp+[0,0,.2], DOWN), other:(standby, DOWN)}, 4., capture)
        errors = self.reach_poses({side:(grasp, DOWN)}, 2.5, capture)
        self.segment(openings={side:0.}, duration=1.5, capture=capture, phase='close')
        self.segment(duration=1., capture=capture, phase='hold')
        forces = self.contact_forces(side)
        a = self.arms[side]
        qf = self.data.qpos[a['fq']]
        result = dict(arm=side, pose_error=errors, contact_N=forces.tolist(),
                      bilateral_contact=bool(np.all(forces > .1)), opening_m=float(qf[0]-qf[1]))
        if not result['bilateral_contact']:
            raise RuntimeError('No bilateral grasp: lift aborted.')
        before = self.lift_monitor.sample(*self.pose(side))
        lift_target = grasp + [0,0,.15]
        q_lift = self.ik(side, lift_target, DOWN)
        self.segment(targets={side:q_lift}, duration=4., capture=capture, phase='lift')
        self.segment(duration=3., capture=capture, phase='hold_lift')
        hold = [dict(time=row['time'], **row[side]) for row in self.log if row['phase']=='hold_lift']
        result.update(assess_lift(before, hold, self.lift_monitor.sample(*self.pose(side))))
        result.update(final_contact_N=self.contact_forces(side).tolist(),
                    final_tcp_error_m=float(np.linalg.norm(self.pose(side)[0]-lift_target)),
                    unexpected_contacts_over_1mm=sorted(self.unexpected_contacts),
                    warnings=[int(w.number) for w in self.data.warning])
        return result

class Recorder:
    def __init__(self, model, path='nero_demo.mp4', fps=25):
        import imageio.v2 as imageio
        self.renderer = mujoco.Renderer(model, height=480, width=640)
        self.camera = mujoco.MjvCamera()
        self.camera.lookat[:] = [0., -.25, .65]
        self.camera.distance, self.camera.azimuth, self.camera.elevation = 2.8, 90, -20
        self.options = mujoco.MjvOption()
        self.options.geomgroup[3] = 0
        self.options.sitegroup[4] = 1
        self.writer = imageio.get_writer(path, fps=fps, codec='libx264', quality=7)
        self.next_frame, self.fps = 0., fps

    def __call__(self, lab):
        if lab.data.time+1e-9 >= self.next_frame:
            mujoco.mj_forward(lab.model, lab.data)
            self.renderer.update_scene(lab.data, camera=self.camera, scene_option=self.options)
            self.writer.append_data(self.renderer.render())
            self.next_frame += 1/self.fps

    def close(self):
        self.writer.close()
        self.renderer.close()

if __name__ == '__main__':
    import argparse
    parser = argparse.ArgumentParser()
    parser.add_argument('--video', action='store_true')
    parser.add_argument('--side', choices=SIDES, default='right')
    args = parser.parse_args()
    build_scene(side=args.side)
    lab = NeroLab()
    recorder = Recorder(lab.model) if args.video else None
    try:
        result = lab.demo(args.side, recorder)
    finally:
        if recorder:
            recorder.close()
    Path('nero_validation.json').write_text(json.dumps(result, indent=2))
    print(json.dumps(result, indent=2))


In [ ]:
from nero_lab import build_scene, NeroLab, Recorder, DOWN
import numpy as np
import mujoco
ARM = 'right'  # 'right' ou 'left' ; relancer depuis cette cellule après modification.
scene_path = build_scene(side=ARM)
lab = NeroLab(scene_path)
print('MuJoCo', mujoco.__version__, '| actionneurs :', lab.model.nu)
for side in ['right', 'left']:
    print(side, 'T_world_tcp :\n', lab.transform(side))

## 4. Approche, fermeture, soulèvement et maintien

La table est à z = 0,40 m, le pot est devant le bras sélectionné. Les deux bras atteignent
d’abord une préprise et une position d’attente. Le bras sélectionné descend, ferme les doigts
sur le pot, puis monte vers une cible 15 cm plus haute pendant 4 s. La pince reste fermée
pendant les 3 s de maintien en hauteur. Le pot reste libre et conserve sa masse de 0,603 kg :
il n’est pas soudé à la pince. Le PD chargé a une petite erreur de suivi ; le pot monte d’environ 14 cm.

`lift_success` exige au moins 10 cm d’élévation réelle du pot, une garde à la table de 8 cm
pendant le maintien, aucun contact avec la table, des contacts bilatéraux sur au moins 99 %
des échantillons et une dérive relative pot/pince inférieure à 1 cm.


In [ ]:
from IPython.display import Video, display
import json
recorder = Recorder(lab.model, 'nero_demo.mp4')
try:
    results = lab.demo(side=ARM, capture=recorder)
finally:
    recorder.close()
print(json.dumps(results, indent=2))
display(Video('nero_demo.mp4', embed=True, width=720))
assert results['bilateral_contact'], 'La pince ne touche pas le pot des deux côtés.'
assert results['lift_success'], 'Soulèvement ou maintien échoué : consulter les mesures.'
assert not any(results['warnings']), 'Avertissement numérique MuJoCo.'
assert not results['unexpected_contacts_over_1mm'], 'Inspecter les contacts supplémentaires.' 

## 5. Mesures d’ouverture, commandes et forces de contact

Une commande moteur limitée n’impose pas la même borne aux pics de force de contact.
La simulation mesure donc séparément la force normale entre chaque doigt et le pot.
Le déplacement réel du pot et sa garde à la table sont également mesurés. Le maintien
est testé pendant 3 s, sans perturbation extérieure.


In [ ]:
import matplotlib.pyplot as plt
t = np.array([row['time'] for row in lab.log])
opening = np.array([row[ARM]['opening'] for row in lab.log])
force = np.array([row[ARM]['force'] for row in lab.log])
contact = np.array([row[ARM]['contact'] for row in lab.log])
clearance = np.array([row[ARM]['clearance_m'] for row in lab.log])
fig, axs = plt.subplots(4, 1, figsize=(10, 10), sharex=True)
axs[0].plot(t, 1000*opening); axs[0].set_ylabel('Ouverture [mm]')
axs[1].plot(t, force[:,0], label='Doigt 1')
axs[1].plot(t, force[:,1], '--', label='Doigt 2')
axs[1].set_ylabel('Commande [N]'); axs[1].legend()
axs[2].plot(t, contact[:,0], label='Contact doigt 1')
axs[2].plot(t, contact[:,1], '--', label='Contact doigt 2')
axs[2].set_ylabel('Normale [N]'); axs[2].legend()
axs[3].plot(t, 100*clearance, label='Bas du pot au-dessus du plateau')
axs[3].axhline(0, color='black', linewidth=.8)
axs[3].set_ylabel('Garde [cm]'); axs[3].legend()
axs[3].set_xlabel('Temps [s]')
for ax in axs: ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 6. Commander des poses indépendantes aux deux bras

Exercice facultatif. `poses` peut contenir un seul bras ou les deux. Pour un seul bras, l’autre
conserve sa consigne. Modifier la position et l’orientation souhaitées dans le repère monde.
La cellule réinitialise la scène, puis garde les pinces ouvertes pendant le mouvement.


In [ ]:
RUN_CUSTOM = False
poses = {
    'right': (np.array([-.25, -.45, .76]), DOWN.copy()),
    'left': (np.array([.45, -.30, .80]), DOWN.copy()),
}
if RUN_CUSTOM:
    lab.reset()
    recorder = Recorder(lab.model, 'nero_custom.mp4')
    try:
        print(lab.reach_poses(poses, duration=4., capture=recorder))
    finally:
        recorder.close()
    print('Contacts supplémentaires :', sorted(lab.unexpected_contacts))
    display(Video('nero_custom.mp4', embed=True, width=720))

# Gains à modifier avant de relancer la démonstration :
# lab.k_finger = 200.       # N/m
# lab.d_finger = 8.         # N.s/m
# lab.force_limit = 6.      # N par doigt, >0 et <=10
# Une fermeture isolée à la pose actuelle :
# lab.segment(openings={'right': 0.}, duration=1.5)

## Validation et limites

Les deux variantes ont été testées localement : élévation du pot d’environ 14 cm, garde finale
d’environ 13,7 cm au-dessus de la table et contact bilatéral conservé pendant 3 s.
Aucun avertissement numérique ni contact inattendu de plus de 1 mm détecté sur ces essais.
Les nombres peuvent varier légèrement selon le rendu et la plateforme.

L’IK d’une cible hors de portée est rejetée. Le notebook ne garantit pas l’absence de collision
pour d’autres poses. La manipulation coordonnée d’un même objet et le transfert
entre pinces restent à développer et tester. Contact-GraspNet n’est pas exécuté ici.

Pour utiliser ensuite ses poses : `T_world_tcp = T_world_camera @ T_camera_grasp @ T_grasp_tcp`.
Identifier les conventions de la sortie du réseau et calibrer la dernière transformation pour **NERO**.

Si OpenGL échoue dans Colab, redémarrer la session puis tout exécuter dans l’ordre, afin que
OSMesa soit configuré avant le premier import MuJoCo. Conserver l’archive avec le notebook.

Sources : votre fichier `nero_dual_arm.mpc.xml` et ses maillages ; la documentation fournie
indique une origine `BhavikShangari/nero-touch` au commit `c6199860a9e9a5c50fc2c93750809d96da4c6819`.
Le [pot French’s YCB](https://github.com/elpis-lab/YCB_Dataset/blob/9e8c6488a2ff673d9aa48a91492fb89423c1b106/ycb/mustard_bottle.xml)
est le même que dans le TP Panda.
